In [1]:
import os
import glob
import pandas as pd
import pickle
import matplotlib.pyplot as plt
import numpy as np
import random
from datetime import datetime, timedelta
from dateutil.relativedelta import relativedelta
import pprint
import pyspark
import pyspark.sql.functions as F

from pyspark.sql.functions import col
from pyspark.sql.types import StringType, IntegerType, FloatType, DateType

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

import xgboost as xgb
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import make_scorer, f1_score, roc_auc_score
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split


In [2]:
# Build a .py script that takes a snapshot date, trains a model and outputs artefact into storage.

## set up pyspark session

In [3]:
# Initialize SparkSession
spark = pyspark.sql.SparkSession.builder \
    .appName("dev") \
    .master("local[*]") \
    .getOrCreate()

# Set log level to ERROR to hide warnings
spark.sparkContext.setLogLevel("ERROR")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/06/19 17:48:27 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## set up config

In [13]:
from datetime import datetime, timedelta
from dateutil.relativedelta import relativedelta
import pprint

# set up config
model_train_date_str = "2024-09-01"  # reference date that marks the end of all historical data
train_test_period_months = 12
oot_period_months = 2
train_test_ratio = 0.8

config = {}
config["model_train_date_str"] = model_train_date_str
config["train_test_period_months"] = train_test_period_months
config["oot_period_months"] = oot_period_months

config["model_train_date"] = datetime.strptime(model_train_date_str, "%Y-%m-%d")
config["oot_end_date"] = config["model_train_date"] - timedelta(days=1)
config["oot_start_date"] = config["model_train_date"] - relativedelta(months=oot_period_months)
config["train_test_end_date"] = config["oot_start_date"] - timedelta(days=1)
config["train_test_start_date"] = config["oot_start_date"] - relativedelta(months=train_test_period_months)
config["train_test_ratio"] = train_test_ratio

# calculate validation period (the month immediately after train_test_end_date)
config["val_start_date"] = config["train_test_end_date"] + timedelta(days=1)
config["val_end_date"] = config["val_start_date"] + relativedelta(months=1) - timedelta(days=1)

# calculate test period (the month immediately after validation)
config["test_start_date"] = config["val_end_date"] + timedelta(days=1)
config["test_end_date"] = config["test_start_date"] + relativedelta(months=1) - timedelta(days=1)

# calculate OOT1 and OOT2
config["oot1_start_date"] = config["model_train_date"]
config["oot1_end_date"] = config["oot1_start_date"] + relativedelta(months=1) - timedelta(days=1)
config["oot2_start_date"] = config["oot1_end_date"] + timedelta(days=1)
config["oot2_end_date"] = config["oot2_start_date"] + relativedelta(months=1) - timedelta(days=1)

# print out all config dates
pprint.pprint(config)

{'model_train_date': datetime.datetime(2024, 9, 1, 0, 0),
 'model_train_date_str': '2024-09-01',
 'oot1_end_date': datetime.datetime(2024, 9, 30, 0, 0),
 'oot1_start_date': datetime.datetime(2024, 9, 1, 0, 0),
 'oot2_end_date': datetime.datetime(2024, 10, 31, 0, 0),
 'oot2_start_date': datetime.datetime(2024, 10, 1, 0, 0),
 'oot_end_date': datetime.datetime(2024, 8, 31, 0, 0),
 'oot_period_months': 2,
 'oot_start_date': datetime.datetime(2024, 7, 1, 0, 0),
 'test_end_date': datetime.datetime(2024, 8, 31, 0, 0),
 'test_start_date': datetime.datetime(2024, 8, 1, 0, 0),
 'train_test_end_date': datetime.datetime(2024, 6, 30, 0, 0),
 'train_test_period_months': 12,
 'train_test_ratio': 0.8,
 'train_test_start_date': datetime.datetime(2023, 7, 1, 0, 0),
 'val_end_date': datetime.datetime(2024, 7, 31, 0, 0),
 'val_start_date': datetime.datetime(2024, 7, 1, 0, 0)}


## get label store

In [14]:
# connect to label store
folder_path = "datamart/gold/label_store/"
files_list = [folder_path+os.path.basename(f) for f in glob.glob(os.path.join(folder_path, '*'))]
label_store_sdf = spark.read.option("header", "true").parquet(*files_list)
print("row_count:",label_store_sdf.count())

label_store_sdf.show()

row_count: 8974
+--------------------+-----------+-----+----------+-------------+
|             loan_id|Customer_ID|label| label_def|snapshot_date|
+--------------------+-----------+-----+----------+-------------+
|CUS_0x1037_2023_0...| CUS_0x1037|    0|30dpd_6mob|   2023-07-01|
|CUS_0x1069_2023_0...| CUS_0x1069|    0|30dpd_6mob|   2023-07-01|
|CUS_0x114a_2023_0...| CUS_0x114a|    0|30dpd_6mob|   2023-07-01|
|CUS_0x1184_2023_0...| CUS_0x1184|    0|30dpd_6mob|   2023-07-01|
|CUS_0x1297_2023_0...| CUS_0x1297|    1|30dpd_6mob|   2023-07-01|
|CUS_0x12fb_2023_0...| CUS_0x12fb|    0|30dpd_6mob|   2023-07-01|
|CUS_0x1325_2023_0...| CUS_0x1325|    0|30dpd_6mob|   2023-07-01|
|CUS_0x1341_2023_0...| CUS_0x1341|    0|30dpd_6mob|   2023-07-01|
|CUS_0x1375_2023_0...| CUS_0x1375|    1|30dpd_6mob|   2023-07-01|
|CUS_0x13a8_2023_0...| CUS_0x13a8|    0|30dpd_6mob|   2023-07-01|
|CUS_0x13ef_2023_0...| CUS_0x13ef|    0|30dpd_6mob|   2023-07-01|
|CUS_0x1440_2023_0...| CUS_0x1440|    0|30dpd_6mob|   2023-0

In [15]:
from pyspark.sql.functions import col

labels_sdf = (
    label_store_sdf
      .filter(
        (col("snapshot_date") >= config["train_test_start_date"]) &
        (col("snapshot_date") <= config["oot2_end_date"])
      )
)

print("extracted labels_sdf count:",
      labels_sdf.count(),
      "from", config["train_test_start_date"].date(),
      "to", config["oot2_end_date"].date())

extracted labels_sdf count: 7985 from 2023-07-01 to 2024-10-31


In [6]:
# extract label store
labels_sdf = label_store_sdf.filter((col("snapshot_date") >= config["train_test_start_date"]) & (col("snapshot_date") <= config["oot_end_date"]))

print("extracted labels_sdf", labels_sdf.count(), config["train_test_start_date"], config["oot_end_date"])

extracted labels_sdf 6961 2023-07-01 00:00:00 2024-08-31 00:00:00


## get features

In [16]:
feature_location = "datamart/gold/feature/"

files_list = [feature_location+os.path.basename(f) for f in glob.glob(os.path.join(feature_location, '*'))]
features_store_sdf = spark.read.option("header", "true").parquet(*files_list)
print("row_count:",features_store_sdf.count())

features_store_sdf.show()


row_count: 218902
+-----------+------+-----+-----+------+------+-----+-----+-----+------+-----+-----+-----+-----+-----+-----+-----+------+-----+-----+------+-------------+-----------+------------------+-----------+-------------+-------+-----------+-------------+---------------------+-----------------+---------------+-------------+-----------+-------------------+----------------------+--------------------+--------------------+----------+----------------+------------------------+---------------------+-------------------+-----------------------+--------------------+---------------+-----------------------------+----------------------+--------------------+---------------------------+-------------------------+---------------+-------------------------+-----------------------------+----------------------+-------------------+-------------------+-----------------+-------------------+------------------+-------------+
|Customer_ID|  fe_1| fe_2| fe_3|  fe_4|  fe_5| fe_6| fe_7| fe_8|  fe_9|fe_10|fe_

In [20]:
from pyspark.sql.functions import col

# this will grab *all* features from the start of train (2023-07-01)
# through the day before model cutoff (2024-08-31)
features_sdf = features_store_sdf.filter(
    (col("snapshot_date") >= config["train_test_start_date"]) &
    (col("snapshot_date") <= config["oot2_end_date"])
)

print(
    "extracted features_sdf count:",
    features_sdf.count(),
    "from", config["train_test_start_date"].date(),
    "to",   config["oot2_end_date"].date()
)

extracted features_sdf count: 145581 from 2023-07-01 to 2024-10-31


## prepare data for modeling

In [26]:
# prepare data for modeling
data_pdf = labels_sdf.join(features_sdf, on=["Customer_ID", "snapshot_date"], how="left").toPandas() #label is the main table then left join with features
data_pdf

,Customer_ID,snapshot_date,loan_id,label,label_def,fe_1,fe_2,fe_3,fe_4,fe_5,...,Auto_Loan_count,Credit_Builder_Loan_count,Debt_Consolidation_Loan_count,Home_Equity_Loan_count,Mortgage_Loan_count,Not_Specified_count,Payday_Loan_count,Personal_Loan_count,Student_Loan_count,Unknown_count
0,CUS_0x1015,2024-02-01,CUS_0x1015_2023_08_01,0,30dpd_6mob,-22.0,128.0,252.0,-46.0,71.0,...,-1,-1,-1,-1,-1,-1,-1,-1,-1,0
1,CUS_0x109d,2024-10-01,CUS_0x109d_2024_04_01,0,30dpd_6mob,107.0,250.0,132.0,294.0,58.0,...,0,0,0,0,2,2,2,0,0,0
2,CUS_0x10eb,2023-09-01,CUS_0x10eb_2023_03_01,0,30dpd_6mob,-26.0,83.0,100.0,16.0,-23.0,...,0,0,0,0,0,0,1,0,1,0
3,CUS_0x112f,2024-04-01,CUS_0x112f_2023_10_01,1,30dpd_6mob,222.0,175.0,140.0,116.0,-63.0,...,1,0,0,0,0,0,0,0,0,0
4,CUS_0x117d,2024-03-01,CUS_0x117d_2023_09_01,0,30dpd_6mob,0.0,90.0,-209.0,-30.0,253.0,...,0,0,1,0,0,1,0,0,2,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7980,CUS_0xc3b9,2024-03-01,CUS_0xc3b9_2023_09_01,0,30dpd_6mob,204.0,-112.0,-164.0,148.0,234.0,...,0,1,0,2,0,1,2,0,1,0
7981,CUS_0xc651,2024-01-01,CUS_0xc651_2023_07_01,0,30dpd_6mob,-71.0,24.0,254.0,99.0,40.0,...,0,0,0,1,1,1,1,0,0,0
7982,CUS_0xf42,2024-10-01,CUS_0xf42_2024_04_01,1,30dpd_6mob,145.0,-40.0,321.0,125.0,55.0,...,0,0,0,0,0,0,0,2,0,0
7983,CUS_0xfb4,2024-06-01,CUS_0xfb4_2023_12_01,0,30dpd_6mob,182.0,148.0,-36.0,151.0,5.0,...,1,0,0,0,1,0,1,0,1,0


In [27]:
import pandas as pd

# make sure snapshot_date is datetime64
data_pdf['snapshot_date'] = pd.to_datetime(data_pdf['snapshot_date'])

# 1) Train: Jul-23 → Jun-24
train_pdf = data_pdf[
    (data_pdf['snapshot_date'] >= config["train_test_start_date"]) &
    (data_pdf['snapshot_date'] <= config["train_test_end_date"])
]

# 2) Val: Jul-24
val_pdf = data_pdf[
    (data_pdf['snapshot_date'] >= config["val_start_date"]) &
    (data_pdf['snapshot_date'] <= config["val_end_date"])
]

# 3) Test: Aug-24
test_pdf = data_pdf[
    (data_pdf['snapshot_date'] >= config["test_start_date"]) &
    (data_pdf['snapshot_date'] <= config["test_end_date"])
]

# 4) OOT1: Sep-24
oot1_pdf = data_pdf[
    (data_pdf['snapshot_date'] >= config["oot1_start_date"]) &
    (data_pdf['snapshot_date'] <= config["oot1_end_date"])
]

# 5) OOT2: Oct-24
oot2_pdf = data_pdf[
    (data_pdf['snapshot_date'] >= config["oot2_start_date"]) &
    (data_pdf['snapshot_date'] <= config["oot2_end_date"])
]

# feature cols
feature_cols = [c for c in data_pdf.columns if c.startswith("fe_")]

# build X / y
X_train, y_train = train_pdf[feature_cols], train_pdf["label"]
X_val,   y_val   = val_pdf[feature_cols],   val_pdf["label"]
X_test,  y_test  = test_pdf[feature_cols],  test_pdf["label"]
X_oot1,  y_oot1  = oot1_pdf[feature_cols],  oot1_pdf["label"]
X_oot2,  y_oot2  = oot2_pdf[feature_cols],  oot2_pdf["label"]

# sanity check
print("Train :", X_train.shape, y_train.mean())
print("Val   :", X_val.shape,   y_val.mean())
print("Test  :", X_test.shape,  y_test.mean())
print("OOT1  :", X_oot1.shape,  y_oot1.mean())
print("OOT2  :", X_oot2.shape,  y_oot2.mean())

Train : (5958, 21) 0.28298086606243705
Val   : (485, 21) 0.29690721649484536
Test  : (518, 21) 0.28378378378378377
OOT1  : (511, 21) 0.3287671232876712
OOT2  : (513, 21) 0.27680311890838205


## preprocess data

In [32]:
from datetime import datetime, timedelta
from dateutil.relativedelta import relativedelta
import pprint
import pandas as pd
from pyspark.sql.functions import col
from sklearn.preprocessing import StandardScaler, OneHotEncoder


drop_cols = [
    'Customer_ID',
    'snapshot_date',
    'label_def',
    'loan_id',
    'Name',
    'SSN',
    'label'
]

feature_cols = [c for c in data_pdf.columns if c not in drop_cols]
print("Using feature columns:", feature_cols)

X_train = train_pdf[feature_cols]; y_train = train_pdf['label']
X_val   = val_pdf[feature_cols];   y_val   = val_pdf['label']
X_test  = test_pdf[feature_cols];  y_test  = test_pdf['label']
X_oot1  = oot1_pdf[feature_cols];  y_oot1  = oot1_pdf['label']
X_oot2  = oot2_pdf[feature_cols];  y_oot2  = oot2_pdf['label']


# ─── 6) ONE‐HOT ENCODE CATEGORICALS ─────────────────────────────────────────────
cat_cols = X_train.select_dtypes(include=['object']).columns.tolist()
num_cols = [c for c in feature_cols if c not in cat_cols]
print("Categorical columns:", cat_cols)
print("Numeric columns:", num_cols)

ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
ohe.fit(X_train[cat_cols])   # fit on train only

def encode_split(X):
    X_cat = pd.DataFrame(
        ohe.transform(X[cat_cols]),
        columns=ohe.get_feature_names_out(cat_cols),
        index=X.index
    )
    X_num = X[num_cols]
    return pd.concat([X_num, X_cat], axis=1)

X_train_enc = encode_split(X_train)
X_val_enc   = encode_split(X_val)
X_test_enc  = encode_split(X_test)
X_oot1_enc  = encode_split(X_oot1)
X_oot2_enc  = encode_split(X_oot2)


# ─── 7) SCALE FEATURES ─────────────────────────────────────────────────────────
scaler = StandardScaler().fit(X_train_enc)  # fit on train only

X_train_processed = scaler.transform(X_train_enc)
X_val_processed   = scaler.transform(X_val_enc)
X_test_processed  = scaler.transform(X_test_enc)
X_oot1_processed  = scaler.transform(X_oot1_enc)
X_oot2_processed  = scaler.transform(X_oot2_enc)

pd.DataFrame(X_train_processed)

Using feature columns: ['fe_1', 'fe_2', 'fe_3', 'fe_4', 'fe_5', 'fe_6', 'fe_7', 'fe_8', 'fe_9', 'fe_10', 'fe_11', 'fe_12', 'fe_13', 'fe_14', 'fe_15', 'fe_16', 'fe_17', 'fe_18', 'fe_19', 'fe_20', 'fe_1_5_mean', 'Occupation', 'Age_num', 'Age_missing', 'Annual_Income', 'Monthly_Inhand_Salary', 'Num_Bank_Accounts', 'Num_Credit_Card', 'Interest_Rate', 'Num_of_Loan', 'Delay_from_due_date', 'Num_of_Delayed_Payment', 'Changed_Credit_Limit', 'Num_Credit_Inquiries', 'Credit_Mix', 'Outstanding_Debt', 'Credit_Utilization_Ratio', 'Payment_of_Min_Amount', 'Total_EMI_per_month', 'Amount_invested_monthly', 'Payment_Behaviour', 'Monthly_Balance', 'days_overdue_per_late_payment', 'Credit_History_Age_num', 'debt_to_income_ratio', 'monthly_repayment_to_income', 'credit_inquiries_per_year', 'Auto_Loan_count', 'Credit_Builder_Loan_count', 'Debt_Consolidation_Loan_count', 'Home_Equity_Loan_count', 'Mortgage_Loan_count', 'Not_Specified_count', 'Payday_Loan_count', 'Personal_Loan_count', 'Student_Loan_count', 

,0,1,2,3,4,5,6,7,8,9,...,73,74,75,76,77,78,79,80,81,82
0,-1.231460,0.259105,1.459434,-1.486085,-0.373797,-0.158732,0.177049,0.587182,1.089140,-0.601635,...,2.665731,-0.755779,-1.026535,-0.394962,-0.475582,-0.350542,-0.341419,-0.387261,1.709479,-0.297686
1,-1.271395,-0.189952,-0.064331,-0.871675,-1.307431,-0.198470,1.459875,-0.272364,1.018259,1.599382,...,-0.375132,1.323137,-1.026535,-0.394962,-0.475582,-0.350542,-0.341419,-0.387261,1.709479,-0.297686
2,1.204572,0.728120,0.336659,0.119309,-1.704722,-1.807863,-1.165443,-0.042485,-0.652500,0.493894,...,-0.375132,1.323137,-1.026535,-0.394962,-0.475582,-0.350542,-0.341419,-0.387261,1.709479,-0.297686
3,-1.011818,-0.120099,-3.161986,-1.327528,1.433877,1.093017,0.177049,-0.262369,1.585305,-1.856513,...,-0.375132,1.323137,-1.026535,-0.394962,-0.475582,2.852724,-0.341419,-0.387261,-0.584974,-0.297686
4,-0.412794,1.386737,0.035916,-2.863553,0.569769,0.238649,-0.618501,-0.072469,0.704359,-0.253057,...,2.665731,-0.755779,-1.026535,-0.394962,-0.475582,-0.350542,-0.341419,2.582238,-0.584974,-0.297686
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5953,-0.452728,0.328958,-0.254802,0.287776,0.460514,-0.009714,-0.260504,-1.441745,-1.503068,0.464016,...,2.665731,-0.755779,-1.026535,-0.394962,-0.475582,-0.350542,-0.341419,-0.387261,1.709479,-0.297686
5954,1.024865,-2.135866,-2.710871,0.436424,1.245163,0.228714,2.126149,0.797071,0.177817,-0.163423,...,-0.375132,-0.755779,0.974151,2.531886,-0.475582,-0.350542,-0.341419,-0.387261,-0.584974,-0.297686
5955,-1.720663,-0.778716,1.479483,-0.049158,-0.681698,-0.744869,-0.071560,1.196860,1.463795,0.663203,...,-0.375132,-0.755779,0.974151,-0.394962,-0.475582,2.852724,-0.341419,-0.387261,-0.584974,-0.297686
5956,0.805222,0.458686,-1.427700,0.466153,-1.029327,-1.559500,1.559319,-1.791560,0.724611,0.135357,...,-0.375132,-0.755779,0.974151,-0.394962,-0.475582,2.852724,-0.341419,-0.387261,-0.584974,-0.297686


## train model

In [37]:
from datetime import datetime, timedelta
from dateutil.relativedelta import relativedelta
import pprint
import pandas as pd
import numpy as np

from pyspark.sql.functions import col

from sklearn.preprocessing     import StandardScaler, OneHotEncoder
from sklearn.compose           import ColumnTransformer
from sklearn.feature_selection import SelectKBest, mutual_info_classif
from sklearn.linear_model      import LogisticRegression
from sklearn.pipeline          import Pipeline
from sklearn.model_selection   import StratifiedKFold, cross_val_score
from sklearn.metrics           import make_scorer, fbeta_score, roc_auc_score, classification_report
from sklearn.metrics           import confusion_matrix

%pip install optuna
import optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 15.8 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 638.8/638.8 kB 9.3 MB/s eta 0:00:00

[notice] A new release of pip is available: 25.0.1 -> 25.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


/usr/local/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [38]:
preproc = ColumnTransformer([
    ("num", StandardScaler(), num_cols),
    ("cat", OneHotEncoder(sparse_output=False, handle_unknown="ignore"), cat_cols),
], remainder="drop")

fbeta_scorer = make_scorer(fbeta_score, beta=2)

# determine number of features after preprocessing
N_FEATURES = preproc.fit_transform(X_train).shape[1]

def objective(trial):
    penalty  = trial.suggest_categorical("penalty", ["l1","l2","elasticnet"])
    C        = trial.suggest_float("C", 1e-3, 1e1, log=True)
    l1_ratio = trial.suggest_float("l1_ratio", 0.0, 1.0) if penalty=="elasticnet" else None
    k        = trial.suggest_int("k", int(0.2*N_FEATURES), N_FEATURES)

    pipe = Pipeline([
        ("pre",    preproc),
        ("select", SelectKBest(mutual_info_classif, k=k)),
        ("lr",     LogisticRegression(
                        solver="saga",
                        penalty=penalty,
                        C=C,
                        l1_ratio=l1_ratio,
                        class_weight="balanced",
                        max_iter=5000,
                        random_state=42
                  ))
    ])

    cv     = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    scores = cross_val_score(pipe, X_train, y_train,
                             cv=cv, scoring=fbeta_scorer, n_jobs=-1)
    return scores.mean()

study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=30)

print("Best F₂ (CV):",   study.best_value)
print("Best params: ",   study.best_trial.params)


# ─── 7) TRAIN FINAL PIPELINE ───────────────────────────────────────────────────
best   = study.best_trial.params
k_best = best["k"]
pipe_final = Pipeline([
    ("pre",    preproc),
    ("select", SelectKBest(mutual_info_classif, k=k_best)),
    ("lr",     LogisticRegression(
                       solver="saga",
                       penalty=best["penalty"],
                       C=best["C"],
                       l1_ratio=best.get("l1_ratio",None),
                       class_weight="balanced",
                       max_iter=5000,
                       random_state=42
                 ))
])
pipe_final.fit(X_train, y_train)


# ─── 8) EVALUATE ON VAL/TEST/OOT ────────────────────────────────────────────────
def gini(y_true, y_proba):
    return 2*roc_auc_score(y_true, y_proba) - 1

for name,(X,y) in [
    ("VAL",  (X_val,  y_val)),
    ("TEST", (X_test, y_test)),
    ("OOT1", (X_oot1, y_oot1)),
    ("OOT2", (X_oot2, y_oot2)),
]:
    p    = pipe_final.predict_proba(X)[:,1]
    yhat = (p>=0.5).astype(int)
    print(f"--- {name} ---")
    print(f"F₂      : {fbeta_score(y, yhat, beta=2):.4f}")
    print(f"ROC AUC : {roc_auc_score(y, p):.4f}")
    print(f"Gini    : {gini(y, p):.4f}")
    print(classification_report(y, yhat, digits=3))


# ─── 9) COMPUTE PSI ON VALIDATION ────────────────────────────────────────────────
def compute_psi(expected, actual, bins):
    eps = 1e-8
    exp_counts = np.histogram(expected, bins=bins)[0] / len(expected)
    act_counts = np.histogram(actual,   bins=bins)[0] / len(actual)
    exp_counts = np.where(exp_counts==0, eps, exp_counts)
    act_counts = np.where(act_counts==0, eps, act_counts)
    return np.sum((act_counts - exp_counts) * np.log(act_counts/exp_counts))

# reference probabilities & decile bins on train
proba_train = pipe_final.predict_proba(X_train)[:,1]
bins = np.unique(np.percentile(proba_train, np.arange(0,101,10)))

# validation probabilities
proba_val = pipe_final.predict_proba(X_val)[:,1]
psi_val  = compute_psi(proba_train, proba_val, bins)
print(f"PSI (Train vs Val): {psi_val:.4f}")


# ─── 10) COMPUTE CSI PER FEATURE ────────────────────────────────────────────────
def compute_csi_per_feature(train_df, val_df, features, n_bins=10):
    eps = 1e-8
    csi = {}
    for feat in features:
        tr = train_df[feat].dropna().values
        vl = val_df[feat].dropna().values
        if np.issubdtype(train_df[feat].dtype, np.number):
            edges    = np.unique(np.percentile(tr, np.linspace(0,100,n_bins+1)))
            if len(edges)<2:
                csi[feat]=0.0; continue
            tr_cnts = np.histogram(tr, bins=edges)[0]/len(tr)
            vl_cnts = np.histogram(vl, bins=edges)[0]/len(vl)
        else:
            cats    = np.unique(np.concatenate([tr,vl]))
            tr_cnts = pd.Series(tr).value_counts(normalize=True).reindex(cats,fill_value=0).values
            vl_cnts = pd.Series(vl).value_counts(normalize=True).reindex(cats,fill_value=0).values
        tr_p = np.where(tr_cnts==0, eps, tr_cnts)
        vl_p = np.where(vl_cnts==0, eps, vl_cnts)
        csi[feat] = np.sum((vl_p-tr_p)*np.log(vl_p/tr_p))
    return pd.Series(csi).sort_values(ascending=False)

# build train/val DataFrames post-preproc (use original splits)
full_train_df = train_pdf[feature_cols]
full_val_df   = val_pdf[feature_cols]
csi_vals = compute_csi_per_feature(full_train_df, full_val_df, feature_cols)
print("CSI per feature (Train→Val):")
print(csi_vals.head(10))

[I 2025-06-19 18:30:28,998] A new study created in memory with name: no-name-726447e0-7f1e-4855-8a48-81b669beb42c
[I 2025-06-19 18:30:31,135] Trial 0 finished with value: 0.6473502950791563 and parameters: {'penalty': 'l2', 'C': 0.1571888844096343, 'k': 79}. Best is trial 0 with value: 0.6473502950791563.
[I 2025-06-19 18:30:32,600] Trial 1 finished with value: 0.6378629639784438 and parameters: {'penalty': 'l2', 'C': 0.4379411186511491, 'k': 56}. Best is trial 0 with value: 0.6473502950791563.
[I 2025-06-19 18:30:33,416] Trial 2 finished with value: 0.6372039898468047 and parameters: {'penalty': 'elasticnet', 'C': 0.026808267609665464, 'l1_ratio': 0.20318384135940337, 'k': 71}. Best is trial 0 with value: 0.6473502950791563.
[I 2025-06-19 18:30:34,156] Trial 3 finished with value: 0.6466279302597953 and parameters: {'penalty': 'l1', 'C': 0.035532015112350275, 'k': 61}. Best is trial 0 with value: 0.6473502950791563.
[I 2025-06-19 18:30:34,693] Trial 4 finished with value: 0.6469005497

Best F₂ (CV): 0.648214003189245
Best params:  {'penalty': 'elasticnet', 'C': 0.07132854235478912, 'l1_ratio': 0.8654632390288842, 'k': 83}
--- VAL ---
F₂      : 0.6439
ROC AUC : 0.7608
Gini    : 0.5217
              precision    recall  f1-score   support

           0      0.847     0.745     0.793       341
           1      0.530     0.681     0.596       144

    accuracy                          0.726       485
   macro avg      0.688     0.713     0.694       485
weighted avg      0.753     0.726     0.734       485

--- TEST ---
F₂      : 0.6675
ROC AUC : 0.7759
Gini    : 0.5517
              precision    recall  f1-score   support

           0      0.869     0.660     0.750       371
           1      0.466     0.748     0.574       147

    accuracy                          0.685       518
   macro avg      0.667     0.704     0.662       518
weighted avg      0.755     0.685     0.700       518

--- OOT1 ---
F₂      : 0.6846
ROC AUC : 0.7750
Gini    : 0.5500
              pr

In [39]:
import numpy as np
from sklearn.metrics import confusion_matrix

# 1) Configuration
threshold = 0.5
C_FP      = 311    # cost per false positive
C_FN      = 7530   # cost per false negative

# 2) Prepare validation data
features = X_val.columns.tolist()
val_df   = X_val.copy()
val_df['label'] = y_val

# 3) Predict probabilities & binary decisions
proba_val = pipe_final.predict_proba(val_df[features])[:, 1]
pred_val  = (proba_val >= threshold).astype(int)

# 4) Compute confusion matrix
tn, fp, fn, tp = confusion_matrix(val_df['label'], pred_val).ravel()

# 5) Compute baseline and current cost
baseline_cost = val_df['label'].sum() * C_FN
current_cost  = fp * C_FP + fn * C_FN

# 6) Average savings per customer
avg_savings_per_cust = (baseline_cost - current_cost) / len(val_df)

print(f"Average savings per customer (Validation): ${avg_savings_per_cust:,.2f}")

Average savings per customer (Validation): $1,465.74


In [41]:
# import optuna
# import numpy as np
# import xgboost as xgb
# import pandas as pd

# from sklearn.compose           import ColumnTransformer
# from sklearn.preprocessing     import StandardScaler, OneHotEncoder
# from sklearn.feature_selection import SelectKBest, mutual_info_classif
# from sklearn.pipeline          import Pipeline
# from sklearn.model_selection   import StratifiedKFold, cross_val_score
# from sklearn.metrics           import (
#     roc_auc_score,
#     fbeta_score,
#     classification_report,
#     make_scorer,
# )
    
# # 1) Preprocessor (reuse your feature splits)
# num_cols = X_train.select_dtypes(include="number").columns.tolist()
# cat_cols = X_train.select_dtypes(include="object").columns.tolist()
# preproc = ColumnTransformer([
#     ("num", StandardScaler(), num_cols),
#     ("cat", OneHotEncoder(sparse_output=False, handle_unknown="ignore"), cat_cols),
# ], remainder="drop")

# # 2) Count post‐preproc features
# X_tr_t     = preproc.fit_transform(X_train)
# N_FEATURES = X_tr_t.shape[1]

# # 3) F2 scorer
# beta         = 2
# f_beta_scorer = make_scorer(fbeta_score, beta=beta)

# # 4) Optuna objective
# def objective(trial):
#     k = trial.suggest_int("k", int(0.2 * N_FEATURES), N_FEATURES)
#     params = {
#         "n_estimators":     trial.suggest_int("n_estimators", 50, 500),
#         "max_depth":        trial.suggest_int("max_depth", 3, 10),
#         "learning_rate":    trial.suggest_float("learning_rate", 1e-3, 0.3, log=True),
#         "subsample":        trial.suggest_float("subsample", 0.6, 1.0),
#         "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
#         "reg_alpha":        trial.suggest_float("reg_alpha", 0.0, 1.0),
#         "reg_lambda":       trial.suggest_float("reg_lambda", 1.0, 3.0),
#         "eval_metric":      "logloss",
#         "random_state":     42,
#         "scale_pos_weight": (y_train == 0).sum() / (y_train == 1).sum(),
#     }
#     pipe = Pipeline([
#         ("pre",    preproc),
#         ("select", SelectKBest(mutual_info_classif, k=k)),
#         ("clf",    xgb.XGBClassifier(**params))
#     ])
#     cv     = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
#     scores = cross_val_score(
#         pipe, X_train, y_train,
#         cv=cv, scoring=f_beta_scorer, n_jobs=-1
#     )
#     return scores.mean()

# # 5) Run the study
# study = optuna.create_study(direction="maximize")
# study.optimize(objective, n_trials=30)

# print(f"▶ Best CV F{beta:.2f}:", study.best_value)
# print("▶ Best params:", study.best_trial.params)

# # 6) Rebuild & fit final pipeline
# best = study.best_trial.params
# pipe_final = Pipeline([
#     ("pre",    preproc),
#     ("select", SelectKBest(mutual_info_classif, k=best["k"])),
#     ("clf",    xgb.XGBClassifier(
#         n_estimators=     best["n_estimators"],
#         max_depth=        best["max_depth"],
#         learning_rate=    best["learning_rate"],
#         subsample=        best["subsample"],
#         colsample_bytree= best["colsample_bytree"],
#         reg_alpha=        best["reg_alpha"],
#         reg_lambda=       best["reg_lambda"],
#         eval_metric=      "logloss",
#         random_state=     42,
#         scale_pos_weight=(y_train == 0).sum() / (y_train == 1).sum(),
#     ))
# ])
# pipe_final.fit(X_train, y_train)

# # 7) Find optimal threshold on VAL
# probas_val = pipe_final.predict_proba(X_val)[:, 1]
# best_thresh, best_f = 0.5, -1
# for t in np.linspace(0, 1, 101):
#     preds = (probas_val >= t).astype(int)
#     fβ    = fbeta_score(y_val, preds, beta=beta)
#     if fβ > best_f:
#         best_f, best_thresh = fβ, t
# print(f"Optimal threshold for F{beta:.2f}: {best_thresh:.2f} → F{beta:.2f} = {best_f:.3f}")

# # 8) Evaluation helper
# def evaluate(split_name, X, y):
#     proba = pipe_final.predict_proba(X)[:, 1]
#     preds = (proba >= best_thresh).astype(int)
#     auc   = roc_auc_score(y, proba)
#     gini  = 2 * auc - 1
#     fβ    = fbeta_score(y, preds, beta=beta)
#     print(f"\n── {split_name} @ thresh={best_thresh:.2f} ──")
#     print(f"  F{beta:.2f} : {fβ:.4f}")
#     print(f"  AUC  : {auc:.4f}")
#     print(f"  Gini : {gini:.4f}")
#     print(classification_report(y, preds, digits=4))

# # 9) Run on all splits
# for name, (X, y) in [
#     ("TRAIN", (X_train, y_train)),
#     ("VAL",   (X_val,   y_val)),
#     ("TEST",  (X_test,  y_test)),
#     ("OOT1",  (X_oot1,  y_oot1)),
#     ("OOT2",  (X_oot2,  y_oot2)),
# ]:
#     evaluate(name, X, y)

# # 10) Which features survived SelectKBest?
# feat_names = pipe_final.named_steps["pre"].get_feature_names_out(feature_cols)
# mask       = pipe_final.named_steps["select"].get_support()
# kept       = feat_names[mask]
# dropped    = feat_names[~mask]

# print(f"\nKept {len(kept)} features:", kept.tolist())
# print(f"Dropped {len(dropped)} features:", dropped.tolist())

In [42]:
import numpy as np
import xgboost as xgb

from sklearn.compose           import ColumnTransformer
from sklearn.preprocessing     import StandardScaler, OneHotEncoder
from sklearn.feature_selection import SelectKBest, mutual_info_classif
from sklearn.pipeline          import Pipeline
from sklearn.metrics           import (
    roc_auc_score,
    fbeta_score,
    classification_report
)

# ── 1) Build the same preprocessor you used for tuning
num_cols = X_train.select_dtypes(include="number").columns.tolist()
cat_cols = X_train.select_dtypes(include="object").columns.tolist()
preproc = ColumnTransformer([
    ("num", StandardScaler(), num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols),
], remainder="drop")

# ── 2) Plug in your best hyperparameters
best_params = {
    "n_estimators":     415,
    "max_depth":        3,
    "learning_rate":    0.009057293740640768,
    "subsample":        0.7276734855131275,
    "colsample_bytree": 0.6232189363962957,
    "reg_alpha":        0.9502195804742386,
    "reg_lambda":       1.539615751287922,
    "eval_metric":      "logloss",
    "random_state":     42,
    "scale_pos_weight": (y_train == 0).sum() / (y_train == 1).sum(),
}
best_k    = 19
beta      = 2.0
best_thresh = 0.32

# ── 3) Build & fit the final pipeline
pipe_final = Pipeline([
    ("pre",    preproc),
    ("select", SelectKBest(mutual_info_classif, k=best_k)),
    ("clf",    xgb.XGBClassifier(**best_params))
])
pipe_final.fit(X_train, y_train)

print(f"Using fixed threshold for F{beta:.0f}: {best_thresh:.2f}")

# ── 4) Evaluation function
def evaluate(split_name, X, y):
    proba = pipe_final.predict_proba(X)[:, 1]
    preds = (proba >= best_thresh).astype(int)
    auc   = roc_auc_score(y, proba)
    gini  = 2 * auc - 1
    f2    = fbeta_score(y, preds, beta=beta)
    print(f"\n{split_name} @ thresh={best_thresh:.2f}")
    print(f"  F{beta:.0f}: {f2:.4f}  AUC: {auc:.4f}  Gini: {gini:.4f}")
    print(classification_report(y, preds, digits=4))

# ── 5) Run on all splits
for split_name, (X, y) in [
    ("TRAIN", (X_train, y_train)),
    ("VAL",   (X_val,   y_val)),
    ("TEST",  (X_test,  y_test)),
    ("OOT1",  (X_oot1,  y_oot1)),
    ("OOT2",  (X_oot2,  y_oot2)),
]:
    evaluate(split_name, X, y)

# ── 6) Which features survived SelectKBest?
feat_names = pipe_final.named_steps["pre"].get_feature_names_out()
mask       = pipe_final.named_steps["select"].get_support()
kept       = feat_names[mask]
dropped    = feat_names[~mask]

print(f"\nKept {len(kept)} features:", kept.tolist())
print(f"Dropped {len(dropped)} features:", dropped.tolist())

Using fixed threshold for F2: 0.32

TRAIN @ thresh=0.32
  F2: 0.7203  AUC: 0.8226  Gini: 0.6451
              precision    recall  f1-score   support

           0     0.9064    0.6051    0.7257      4272
           1     0.4569    0.8416    0.5922      1686

    accuracy                         0.6720      5958
   macro avg     0.6816    0.7234    0.6590      5958
weighted avg     0.7792    0.6720    0.6879      5958


VAL @ thresh=0.32
  F2: 0.7324  AUC: 0.8017  Gini: 0.6035
              precision    recall  f1-score   support

           0     0.9021    0.6217    0.7361       341
           1     0.4840    0.8403    0.6142       144

    accuracy                         0.6866       485
   macro avg     0.6931    0.7310    0.6752       485
weighted avg     0.7780    0.6866    0.6999       485


TEST @ thresh=0.32
  F2: 0.6930  AUC: 0.7847  Gini: 0.5693
              precision    recall  f1-score   support

           0     0.8884    0.5580    0.6854       371
           1     0.424

In [43]:
import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix, fbeta_score

# ─── 1) Average savings per customer on Validation ──────────────────────────────
threshold = best_thresh
C_FP      = 311    # cost per false positive
C_FN      = 7530   # cost per false negative

# Predict on Val
proba_val = pipe_final.predict_proba(X_val)[:, 1]
pred_val  = (proba_val >= threshold).astype(int)

# Confusion matrix
tn, fp, fn, tp = confusion_matrix(y_val, pred_val).ravel()

# Compute costs & savings
baseline_cost        = y_val.sum() * C_FN
current_cost         = fp * C_FP + fn * C_FN
avg_savings_per_cust = (baseline_cost - current_cost) / len(y_val)

print(f"Average savings per customer (Validation): ${avg_savings_per_cust:,.2f}")

# ─── 2) Population Stability Index (PSI) ─────────────────────────────────────────
def compute_psi(expected, actual, bins):
    eps = 1e-8
    exp_freq = np.histogram(expected, bins=bins)[0] / len(expected)
    act_freq = np.histogram(actual,   bins=bins)[0] / len(actual)
    exp_freq = np.where(exp_freq==0, eps, exp_freq)
    act_freq = np.where(act_freq==0, eps, act_freq)
    return np.sum((act_freq - exp_freq) * np.log(act_freq/exp_freq))

# reference distribution on TRAIN
proba_train = pipe_final.predict_proba(X_train)[:, 1]
bin_edges   = np.unique(np.percentile(proba_train, np.arange(0, 101, 10)))

psi_val = compute_psi(proba_train, proba_val, bin_edges)
print(f"PSI (Train → Validation): {psi_val:.4f}")

# ─── 3) Characteristic Stability Index (CSI) per feature ────────────────────────
def compute_csi_per_feature(train_df, val_df, features, n_bins=10):
    eps = 1e-8
    csi = {}
    for feat in features:
        tr = train_df[feat].dropna().values
        vl = val_df[feat].dropna().values

        if np.issubdtype(train_df[feat].dtype, np.number):
            # numeric: bin by train-deciles
            edges = np.unique(np.percentile(tr, np.linspace(0,100,n_bins+1)))
            if len(edges) < 2:
                csi[feat] = 0.0
                continue
            tr_cnt = np.histogram(tr, bins=edges)[0] / len(tr)
            vl_cnt = np.histogram(vl, bins=edges)[0] / len(vl)
        else:
            # categorical: compare category proportions
            cats    = np.unique(np.concatenate([tr, vl]))
            tr_cnt  = pd.Series(tr).value_counts(normalize=True).reindex(cats, fill_value=0).values
            vl_cnt  = pd.Series(vl).value_counts(normalize=True).reindex(cats, fill_value=0).values

        tr_p = np.where(tr_cnt==0, eps, tr_cnt)
        vl_p = np.where(vl_cnt==0, eps, vl_cnt)
        csi[feat] = np.sum((vl_p - tr_p) * np.log(vl_p / tr_p))

    return pd.Series(csi, name="CSI_Validation").sort_values(ascending=False)

# build raw train/val DataFrames
train_raw = train_pdf[feature_cols]
val_raw   = val_pdf[feature_cols]

csi_vals = compute_csi_per_feature(train_raw, val_raw, feature_cols)
print("\nTop 10 CSI on Validation:")
print(csi_vals.head(10).round(4))

Average savings per customer (Validation): $1,795.90
PSI (Train → Validation): 0.0210

Top 10 CSI on Validation:
Occupation                       0.0487
fe_17                            0.0464
fe_15                            0.0442
Changed_Credit_Limit             0.0394
days_overdue_per_late_payment    0.0334
fe_11                            0.0314
Amount_invested_monthly          0.0305
fe_9                             0.0292
Monthly_Inhand_Salary            0.0290
Annual_Income                    0.0288
Name: CSI_Validation, dtype: float64


## prepare model artefact to save

In [44]:
# ─── 1) Initialize artefact dict ────────────────────────────────────────────────
model_artefact = {}

# Store the fitted pipeline
model_artefact['model'] = pipe_final

# Version string (updated prefix)
model_artefact['model_version'] = \
    "xgboostv1_" + config["model_train_date_str"].replace("-", "_")

# Keep your preprocessing pieces for later re-use
model_artefact['preprocessing_transformers'] = {
    "preprocessor":        preproc,
    "feature_selector":    pipe_final.named_steps["select"]
}

# Record the data‐cutoff dates
model_artefact['data_dates'] = config

# ─── 2) Data stats ──────────────────────────────────────────────────────────────
model_artefact['data_stats'] = {
    "X_train_rows":  X_train.shape[0],
    "X_val_rows":    X_val.shape[0],
    "X_test_rows":   X_test.shape[0],
    "X_oot1_rows":   X_oot1.shape[0],
    "X_oot2_rows":   X_oot2.shape[0],
    "y_train_rate":  round(y_train.mean(), 4),
    "y_val_rate":    round(y_val.mean(),   4),
    "y_test_rate":   round(y_test.mean(),  4),
    "y_oot1_rate":   round(y_oot1.mean(),  4),
    "y_oot2_rate":   round(y_oot2.mean(),  4),
}

# ─── 3) Compute metrics for each split ──────────────────────────────────────────
def compute_metrics(X, y, threshold):
    proba = pipe_final.predict_proba(X)[:, 1]
    preds = (proba >= threshold).astype(int)
    f2    = fbeta_score(y, preds, beta=beta)
    auc   = roc_auc_score(y, proba)
    gini  = 2 * auc - 1
    tn, fp, fn, tp = confusion_matrix(y, preds).ravel()
    return {
        "f2":   round(f2,   4),
        "auc":  round(auc,  4),
        "gini": round(gini, 4),
        "fp":   int(fp),
        "fn":   int(fn),
        "tp":   int(tp),
        "tn":   int(tn),
    }

thr = best_thresh
metrics = {
    "train": compute_metrics(X_train, y_train, thr),
    "val":   compute_metrics(X_val,   y_val,   thr),
    "test":  compute_metrics(X_test,  y_test,  thr),
    "oot1":  compute_metrics(X_oot1,  y_oot1,  thr),
    "oot2":  compute_metrics(X_oot2,  y_oot2,  thr),
}
model_artefact['results'] = {"metrics": metrics}

# ─── 4) Add avg savings, PSI & CSI ─────────────────────────────────────────────
C_FP = 311
C_FN = 7530
fp = metrics["val"]["fp"]
fn = metrics["val"]["fn"]
baseline_cost = y_val.sum() * C_FN
current_cost  = fp * C_FP + fn * C_FN
avg_sav = (baseline_cost - current_cost) / len(y_val)

model_artefact['results']['avg_savings_per_cust_val'] = round(avg_sav, 2)
model_artefact['results']['psi_val']                 = round(psi_val, 4)
model_artefact['results']['csi_val']                 = csi_vals.to_dict()

# ─── 5) Hyperparameters ─────────────────────────────────────────────────────────
model_artefact['hp_params'] = study.best_trial.params

# ─── 6) Pretty‐print everything ────────────────────────────────────────────────
import pprint
pprint.pprint(model_artefact, width=120)

{'data_dates': {'model_train_date': datetime.datetime(2024, 9, 1, 0, 0),
                'model_train_date_str': '2024-09-01',
                'oot1_end_date': datetime.datetime(2024, 9, 30, 0, 0),
                'oot1_start_date': datetime.datetime(2024, 9, 1, 0, 0),
                'oot2_end_date': datetime.datetime(2024, 10, 31, 0, 0),
                'oot2_start_date': datetime.datetime(2024, 10, 1, 0, 0),
                'oot_end_date': datetime.datetime(2024, 8, 31, 0, 0),
                'oot_period_months': 2,
                'oot_start_date': datetime.datetime(2024, 7, 1, 0, 0),
                'test_end_date': datetime.datetime(2024, 8, 31, 0, 0),
                'test_start_date': datetime.datetime(2024, 8, 1, 0, 0),
                'train_test_end_date': datetime.datetime(2024, 6, 30, 0, 0),
                'train_test_period_months': 12,
                'train_test_ratio': 0.8,
                'train_test_start_date': datetime.datetime(2023, 7, 1, 0, 0),
                '

## save artefact to model bank

In [45]:
# create model_bank dir
model_bank_directory = "model_bank/"

if not os.path.exists(model_bank_directory):
    os.makedirs(model_bank_directory)

In [46]:
import os
import pickle

# Choose any filename you prefer
filename = "xgboostv1.pkl"

# Full path to the file
file_path = os.path.join(model_bank_directory, filename)

# Write the model to a pickle file
with open(file_path, "wb") as f:
    pickle.dump(model_artefact, f)

print(f"Model saved to {file_path}")

Model saved to model_bank/xgboostv1.pkl


In [15]:
# # Full path to the file
# file_path = os.path.join(model_bank_directory, model_artefact['model_version'] + '.pkl')

# # Write the model to a pickle file
# with open(file_path, 'wb') as file:
#     pickle.dump(model_artefact, file)

# print(f"Model saved to {file_path}")


Model saved to model_bank/credit_model_2024_09_01.pkl


## test load pickle and make model inference

In [50]:
import pprint

# 1) Load the artefact 
filename = "xgboostv1.pkl"
file_path = os.path.join(model_bank_directory, filename)
with open(file_path, "rb") as f:
    arte = pickle.load(f)

# 2) Model version & data dates
print("Model version:", arte.get("model_version", "<none>"))
print("Data cutoffs:")
pprint.pprint(arte["data_dates"])

# 3) Data stats
print("\nData stats:")
for k, v in arte["data_stats"].items():
    print(f"  {k}: {v}")

# 4) Per‐split metrics
print("\nSplit metrics:")
metrics = arte["results"]["metrics"]
for split, m in metrics.items():
    print(f"  {split.upper()}:")
    print(f"    F2    : {m['f2']}")
    print(f"    AUC   : {m['auc']}")
    print(f"    Gini  : {m['gini']}")
    print(f"    TP/FP/TN/FN: {m['tp']}/{m['fp']}/{m['tn']}/{m['fn']}")

# 5) Avg savings, PSI, CSI on Validation
print(f"\nAverage savings per customer (Validation): ${arte['results']['avg_savings_per_cust_val']:.2f}")
print(f"PSI (Train→Validation): {arte['results']['psi_val']:.4f}")

print("\nCSI per feature (Validation):")
for feat, val in arte["results"]["csi_val"].items():
    print(f"  {feat}: {val:.4f}")


Model version: xgboostv1_2024_09_01
Data cutoffs:
{'model_train_date': datetime.datetime(2024, 9, 1, 0, 0),
 'model_train_date_str': '2024-09-01',
 'oot1_end_date': datetime.datetime(2024, 9, 30, 0, 0),
 'oot1_start_date': datetime.datetime(2024, 9, 1, 0, 0),
 'oot2_end_date': datetime.datetime(2024, 10, 31, 0, 0),
 'oot2_start_date': datetime.datetime(2024, 10, 1, 0, 0),
 'oot_end_date': datetime.datetime(2024, 8, 31, 0, 0),
 'oot_period_months': 2,
 'oot_start_date': datetime.datetime(2024, 7, 1, 0, 0),
 'test_end_date': datetime.datetime(2024, 8, 31, 0, 0),
 'test_start_date': datetime.datetime(2024, 8, 1, 0, 0),
 'train_test_end_date': datetime.datetime(2024, 6, 30, 0, 0),
 'train_test_period_months': 12,
 'train_test_ratio': 0.8,
 'train_test_start_date': datetime.datetime(2023, 7, 1, 0, 0),
 'val_end_date': datetime.datetime(2024, 7, 31, 0, 0),
 'val_start_date': datetime.datetime(2024, 7, 1, 0, 0)}

Data stats:
  X_train_rows: 5958
  X_val_rows: 485
  X_test_rows: 518
  X_oot1_r